# Comprendre l'Overfitting en Trading Algorithmique

## Section 0 : Setup

In [ ]:
import numpy as np
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column, row, gridplot
from bokeh.models import (
    ColumnDataSource, Slider, CustomJS, Span, Label,
    HoverTool, ColorBar, LinearColorMapper, BoxAnnotation,
    Div, Toggle, Range1d, Title
)
from bokeh.io import curdoc
from bokeh.palettes import RdYlGn11
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

# Configuration Bokeh
output_notebook()

# Constantes de style
DARK_BG = '#1a1a1a'
BORDER_COLOR = '#1a1a1a'
GRID_COLOR = '#3a3a3a'
TEXT_COLOR = '#c0c0c0'
GREEN_GOOD = '#00FF00'
RED_BAD = '#FF4444'
YELLOW_DATA = '#FFD700'
PLOT_WIDTH = 900
PLOT_HEIGHT = 350

def apply_dark_theme(p):
    """Applique le theme sombre a une figure Bokeh."""
    p.background_fill_color = DARK_BG
    p.border_fill_color = BORDER_COLOR
    p.outline_line_color = '#333333'
    p.grid.grid_line_color = GRID_COLOR
    p.grid.grid_line_alpha = 0.3
    p.xaxis.axis_label_text_color = TEXT_COLOR
    p.yaxis.axis_label_text_color = TEXT_COLOR
    p.xaxis.major_label_text_color = TEXT_COLOR
    p.yaxis.major_label_text_color = TEXT_COLOR
    p.title.text_color = TEXT_COLOR
    p.legend.background_fill_color = DARK_BG
    p.legend.label_text_color = TEXT_COLOR
    p.legend.border_line_color = '#333333'
    return p

print("Setup complete!")

## Section 1 : Le Costume Sur-Mesure

In [ ]:
# Analogie du costume sur-mesure
# Plus on ajoute de reglages precis, plus le costume devient rigide

# Forme simplifiee d'un costume (corps + manches)
body_x = [0, 0.3, 0.35, 0.5, 0.65, 0.7, 1, 0.85, 0.8, 0.7, 0.3, 0.2, 0.15, 0]
body_y = [0.3, 0.3, 0.5, 0.55, 0.5, 0.3, 0.3, 0.8, 0.85, 1, 1, 0.85, 0.8, 0.3]

# Source de donnees
source = ColumnDataSource(data=dict(
    x=[body_x],
    y=[body_y],
    color=[GREEN_GOOD]
))

# Figure
p1 = figure(title="Le Costume Sur-Mesure", width=500, height=500,
            x_range=(-0.2, 1.2), y_range=(0, 1.2),
            tools="")
apply_dark_theme(p1)
p1.xaxis.visible = False
p1.yaxis.visible = False
p1.grid.visible = False

# Dessin du costume
costume = p1.patches(xs='x', ys='y', fill_color='color', 
                     line_color='white', line_width=2, source=source, alpha=0.8)

# Sliders pour les ajustements
slider_col = Slider(start=0, end=10, value=0, step=1, title="Precision du col", width=200)
slider_manches = Slider(start=0, end=10, value=0, step=1, title="Precision des manches", width=200)
slider_taille = Slider(start=0, end=10, value=0, step=1, title="Precision de la taille", width=200)
slider_epaules = Slider(start=0, end=10, value=0, step=1, title="Precision des epaules", width=200)

# Div pour le message et le compteur
message_div = Div(text="""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin-top: 10px;'>
    <h3 style='color: #00FF00; margin: 0;'>Ajustements: 0 / 40</h3>
    <div style='background: #3a3a3a; border-radius: 10px; height: 20px; margin: 10px 0;'>
        <div id='progress' style='background: #00FF00; height: 100%; width: 0%; border-radius: 10px; transition: all 0.3s;'></div>
    </div>
    <p style='color: #c0c0c0; margin: 10px 0 0 0;'>Un costume flexible s'adapte a differentes morphologies.</p>
</div>
""", width=500)

# Callback JavaScript
callback = CustomJS(args=dict(source=source, div=message_div,
                              s1=slider_col, s2=slider_manches, 
                              s3=slider_taille, s4=slider_epaules), code="""
    const total = s1.value + s2.value + s3.value + s4.value;
    const max_total = 40;
    const ratio = total / max_total;
    
    // Interpolation de couleur vert -> jaune -> rouge
    let color;
    if (ratio < 0.5) {
        // Vert vers Jaune
        const r = Math.round(255 * (ratio * 2));
        const g = 255;
        color = `rgb(${r}, ${g}, 0)`;
    } else {
        // Jaune vers Rouge
        const r = 255;
        const g = Math.round(255 * (1 - (ratio - 0.5) * 2));
        color = `rgb(${r}, ${g}, 0)`;
    }
    
    source.data['color'] = [color];
    source.change.emit();
    
    // Message dynamique
    let message, title_color;
    if (ratio < 0.25) {
        message = "Un costume flexible s'adapte a differentes morphologies.";
        title_color = '#00FF00';
    } else if (ratio < 0.5) {
        message = "Le costume commence a etre plus specifique...";
        title_color = '#AAFF00';
    } else if (ratio < 0.75) {
        message = "Attention! Le costume devient rigide et peu adaptable.";
        title_color = '#FFAA00';
    } else {
        message = "DANGER! Ce costume ne convient QU'A UNE SEULE personne. C'est l'OVERFITTING!";
        title_color = '#FF4444';
    }
    
    div.text = `
    <div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin-top: 10px;'>
        <h3 style='color: ${title_color}; margin: 0;'>Ajustements: ${total} / ${max_total}</h3>
        <div style='background: #3a3a3a; border-radius: 10px; height: 20px; margin: 10px 0;'>
            <div style='background: ${color}; height: 100%; width: ${ratio * 100}%; border-radius: 10px; transition: all 0.3s;'></div>
        </div>
        <p style='color: #c0c0c0; margin: 10px 0 0 0;'>${message}</p>
    </div>
    `;
""")

slider_col.js_on_change('value', callback)
slider_manches.js_on_change('value', callback)
slider_taille.js_on_change('value', callback)
slider_epaules.js_on_change('value', callback)

# Layout
sliders = column(slider_col, slider_manches, slider_taille, slider_epaules)
layout1 = row(column(p1, message_div), sliders)

show(layout1)

## Section 2 : Signal vs Bruit

In [ ]:
# Generation des donnees synthetiques
np.random.seed(42)
n_points = 50
x = np.linspace(0, 10, n_points)

# Signal reel (relation lineaire)
signal = 2 * x + 5

# Bruit (fluctuations aleatoires)
noise_level = 0.3
noise = np.random.normal(0, noise_level * np.std(signal), n_points)

# Donnees observees
observed = signal + noise

# Sources de donnees
source_signal = ColumnDataSource(data=dict(x=x, y=signal))
source_noise = ColumnDataSource(data=dict(x=x, y=noise))
source_observed = ColumnDataSource(data=dict(x=x, y=observed))

# Figure 1: Signal
p_signal = figure(title="SIGNAL - La vraie relation (y = 2x + 5)", 
                  width=PLOT_WIDTH, height=250,
                  x_axis_label="x", y_axis_label="y")
apply_dark_theme(p_signal)
p_signal.line('x', 'y', source=source_signal, color=GREEN_GOOD, line_width=3, legend_label="Signal")
p_signal.legend.location = "top_left"

# Figure 2: Bruit
p_noise = figure(title="BRUIT - Fluctuations aleatoires", 
                 width=PLOT_WIDTH, height=250,
                 x_range=p_signal.x_range,
                 x_axis_label="x", y_axis_label="bruit")
apply_dark_theme(p_noise)
p_noise.scatter('x', 'y', source=source_noise, color=RED_BAD, size=8, alpha=0.7, legend_label="Bruit")
zero_line = Span(location=0, dimension='width', line_color='white', line_dash='dashed', line_width=1)
p_noise.add_layout(zero_line)
p_noise.legend.location = "top_left"

# Figure 3: Donnees observees
p_observed = figure(title="DONNEES OBSERVEES - Signal + Bruit (ce qu'on voit en realite)", 
                    width=PLOT_WIDTH, height=250,
                    x_range=p_signal.x_range,
                    x_axis_label="x", y_axis_label="y")
apply_dark_theme(p_observed)
p_observed.scatter('x', 'y', source=source_observed, color=YELLOW_DATA, size=10, alpha=0.8, legend_label="Donnees")
p_observed.line('x', 'y', source=source_signal, color=GREEN_GOOD, line_width=2, line_dash='dashed', 
                alpha=0.5, legend_label="Signal cache")
p_observed.legend.location = "top_left"

# Slider pour le niveau de bruit
noise_slider = Slider(start=0.1, end=1.0, value=0.3, step=0.1, title="Niveau de bruit", width=400)

# Callback pour regenerer le bruit
noise_callback = CustomJS(args=dict(source_noise=source_noise, source_observed=source_observed,
                                    source_signal=source_signal, slider=noise_slider), code="""
    const noise_level = slider.value;
    const x = source_signal.data['x'];
    const signal = source_signal.data['y'];
    const n = x.length;
    
    // Calcul de l'ecart-type du signal
    const mean_signal = signal.reduce((a, b) => a + b, 0) / n;
    const std_signal = Math.sqrt(signal.map(s => (s - mean_signal) ** 2).reduce((a, b) => a + b, 0) / n);
    
    // Generation du nouveau bruit (Box-Muller)
    const noise = [];
    const observed = [];
    for (let i = 0; i < n; i++) {
        const u1 = Math.random();
        const u2 = Math.random();
        const z = Math.sqrt(-2 * Math.log(u1)) * Math.cos(2 * Math.PI * u2);
        const n_val = z * noise_level * std_signal;
        noise.push(n_val);
        observed.push(signal[i] + n_val);
    }
    
    source_noise.data['y'] = noise;
    source_observed.data['y'] = observed;
    source_noise.change.emit();
    source_observed.change.emit();
""")

noise_slider.js_on_change('value', noise_callback)

# Explication
explanation_div = Div(text="""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin: 10px 0;'>
    <p style='color: #c0c0c0; margin: 0;'>
        <span style='color: #00FF00;'>Le SIGNAL</span> est la vraie relation sous-jacente que nous cherchons a capturer.<br>
        <span style='color: #FF4444;'>Le BRUIT</span> represente les fluctuations aleatoires (hasard, erreurs, evenements imprevisibles).<br>
        <span style='color: #FFD700;'>Les DONNEES</span> que nous observons sont toujours un melange des deux.
    </p>
</div>
""", width=PLOT_WIDTH)

# Layout
layout2 = column(explanation_div, noise_slider, p_signal, p_noise, p_observed)
show(layout2)

## Section 3 : Regression Polynomiale

In [ ]:
# Precalcul de toutes les regressions polynomiales
max_degree = 15
x_dense = np.linspace(0, 10, 200)  # Points denses pour les courbes lisses

polynomial_fits = {}
for degree in range(1, max_degree + 1):
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(x.reshape(-1, 1))
    X_dense_poly = poly.transform(x_dense.reshape(-1, 1))
    
    model = LinearRegression()
    model.fit(X_poly, observed)
    
    y_pred_dense = model.predict(X_dense_poly)
    y_pred = model.predict(X_poly)
    mse = mean_squared_error(observed, y_pred)
    
    polynomial_fits[degree] = {
        'y_dense': y_pred_dense.tolist(),
        'mse': float(mse)
    }

# Source pour la courbe de regression
source_poly = ColumnDataSource(data=dict(
    x=x_dense.tolist(),
    y=polynomial_fits[1]['y_dense']
))

# Figure
p3 = figure(title="Regression Polynomiale - Degre 1",
            width=PLOT_WIDTH, height=PLOT_HEIGHT,
            x_axis_label="x", y_axis_label="y")
apply_dark_theme(p3)

# Donnees
p3.scatter(x, observed, color=YELLOW_DATA, size=10, alpha=0.8, legend_label="Donnees")

# Signal reel
p3.line(x, signal, color=GREEN_GOOD, line_width=2, line_dash='dashed', 
        alpha=0.5, legend_label="Signal reel")

# Regression
poly_line = p3.line('x', 'y', source=source_poly, color='#00BFFF', line_width=3, 
                    legend_label="Regression")

p3.legend.location = "top_left"

# Slider pour le degre
degree_slider = Slider(start=1, end=15, value=1, step=1, title="Degre du polynome", width=400)

# Div pour les metriques
metrics_div = Div(text=f"""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; display: flex; gap: 30px;'>
    <div>
        <span style='color: #c0c0c0;'>Degre:</span>
        <span style='color: #00BFFF; font-size: 24px; font-weight: bold;'>1</span>
    </div>
    <div>
        <span style='color: #c0c0c0;'>MSE (erreur):</span>
        <span style='color: #00FF00; font-size: 24px; font-weight: bold;'>{polynomial_fits[1]['mse']:.4f}</span>
    </div>
    <div>
        <span style='color: #c0c0c0;'>Status:</span>
        <span style='color: #00FF00; font-size: 18px;'>Capture le signal</span>
    </div>
</div>
""", width=PLOT_WIDTH)

# Stocker les fits dans un format JSON-compatible
fits_data = {str(k): v for k, v in polynomial_fits.items()}

# Callback
degree_callback = CustomJS(args=dict(source=source_poly, div=metrics_div, 
                                     slider=degree_slider, fits=fits_data,
                                     title=p3.title), code="""
    const degree = slider.value;
    const fit = fits[degree.toString()];
    
    source.data['y'] = fit.y_dense;
    source.change.emit();
    
    title.text = `Regression Polynomiale - Degre ${degree}`;
    
    let status, status_color;
    if (degree <= 2) {
        status = "Capture le signal";
        status_color = '#00FF00';
    } else if (degree <= 5) {
        status = "Bon compromis";
        status_color = '#AAFF00';
    } else if (degree <= 8) {
        status = "Commence a capturer le bruit";
        status_color = '#FFAA00';
    } else {
        status = "OVERFITTING! Capture le bruit";
        status_color = '#FF4444';
    }
    
    const mse_color = fit.mse < 20 ? '#00FF00' : (fit.mse < 50 ? '#FFAA00' : '#FF4444');
    
    div.text = `
    <div style='background: #2a2a2a; padding: 15px; border-radius: 5px; display: flex; gap: 30px;'>
        <div>
            <span style='color: #c0c0c0;'>Degre:</span>
            <span style='color: #00BFFF; font-size: 24px; font-weight: bold;'>${degree}</span>
        </div>
        <div>
            <span style='color: #c0c0c0;'>MSE (erreur):</span>
            <span style='color: ${mse_color}; font-size: 24px; font-weight: bold;'>${fit.mse.toFixed(4)}</span>
        </div>
        <div>
            <span style='color: #c0c0c0;'>Status:</span>
            <span style='color: ${status_color}; font-size: 18px;'>${status}</span>
        </div>
    </div>
    `;
""")

degree_slider.js_on_change('value', degree_callback)

# Explication
poly_explanation = Div(text="""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin-bottom: 10px;'>
    <p style='color: #c0c0c0; margin: 0;'>
        <b>Degre 1-2:</b> Le modele capture la tendance generale (le signal).<br>
        <b>Degre 3-5:</b> Le modele s'ajuste mieux mais reste generaliste.<br>
        <b>Degre 6+:</b> Le modele commence a passer par chaque point = il memorise le bruit!<br><br>
        <span style='color: #FF4444;'>Plus le degre augmente, plus l'erreur sur les donnees d'entrainement diminue...</span><br>
        <span style='color: #FF4444;'>mais ca ne veut pas dire que le modele est meilleur!</span>
    </p>
</div>
""", width=PLOT_WIDTH)

layout3 = column(poly_explanation, degree_slider, metrics_div, p3)
show(layout3)

## Section 4 : Train/Test Split

In [ ]:
# Split des donnees
train_ratio = 0.7
split_idx = int(len(x) * train_ratio)

x_train, x_test = x[:split_idx], x[split_idx:]
y_train, y_test = observed[:split_idx], observed[split_idx:]

# Calcul des erreurs pour chaque degre
train_errors = []
test_errors = []
degrees = list(range(1, max_degree + 1))

for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(x_train.reshape(-1, 1))
    X_test_poly = poly.transform(x_test.reshape(-1, 1))
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    train_pred = model.predict(X_train_poly)
    test_pred = model.predict(X_test_poly)
    
    train_errors.append(mean_squared_error(y_train, train_pred))
    test_errors.append(mean_squared_error(y_test, test_pred))

# Limiter les erreurs extremes pour la visualisation
test_errors_capped = [min(e, 500) for e in test_errors]

# Source de donnees
source_errors = ColumnDataSource(data=dict(
    degree=degrees,
    train=train_errors,
    test=test_errors_capped
))

# Figure des erreurs
p4_error = figure(title="Erreur Train vs Test selon la complexite du modele",
                  width=PLOT_WIDTH, height=350,
                  x_axis_label="Degre du polynome", y_axis_label="Erreur (MSE)",
                  y_range=(0, 200))
apply_dark_theme(p4_error)

# Zones colorees
underfitting_zone = BoxAnnotation(left=0.5, right=2.5, fill_alpha=0.1, fill_color='#3498db')
optimal_zone = BoxAnnotation(left=2.5, right=5.5, fill_alpha=0.1, fill_color=GREEN_GOOD)
overfitting_zone = BoxAnnotation(left=5.5, right=15.5, fill_alpha=0.1, fill_color=RED_BAD)

p4_error.add_layout(underfitting_zone)
p4_error.add_layout(optimal_zone)
p4_error.add_layout(overfitting_zone)

# Courbes d'erreur
p4_error.line('degree', 'train', source=source_errors, color=GREEN_GOOD, line_width=3, 
              legend_label="Erreur Train")
p4_error.scatter('degree', 'train', source=source_errors, color=GREEN_GOOD, size=10)

p4_error.line('degree', 'test', source=source_errors, color=RED_BAD, line_width=3, 
              legend_label="Erreur Test")
p4_error.scatter('degree', 'test', source=source_errors, color=RED_BAD, size=10)

p4_error.legend.location = "top_left"

# Labels pour les zones
p4_error.add_layout(Label(x=1.5, y=180, text="UNDERFITTING", text_color='#3498db', 
                          text_font_size='12px', text_font_style='bold'))
p4_error.add_layout(Label(x=3.5, y=180, text="OPTIMAL", text_color=GREEN_GOOD, 
                          text_font_size='12px', text_font_style='bold'))
p4_error.add_layout(Label(x=9, y=180, text="OVERFITTING", text_color=RED_BAD, 
                          text_font_size='12px', text_font_style='bold'))

# Figure des donnees avec split visible
p4_data = figure(title="Donnees: Train (70%) vs Test (30%)",
                 width=PLOT_WIDTH, height=250,
                 x_axis_label="x", y_axis_label="y")
apply_dark_theme(p4_data)

# Zone de split
split_zone = BoxAnnotation(left=x[split_idx], right=10, fill_alpha=0.15, fill_color=RED_BAD)
p4_data.add_layout(split_zone)

p4_data.scatter(x_train, y_train, color=GREEN_GOOD, size=10, alpha=0.8, legend_label="Train")
p4_data.scatter(x_test, y_test, color=RED_BAD, size=10, alpha=0.8, legend_label="Test")

# Ligne de separation
split_line = Span(location=x[split_idx], dimension='height', line_color='white', 
                  line_dash='dashed', line_width=2)
p4_data.add_layout(split_line)

p4_data.legend.location = "top_left"

# Explication
split_explanation = Div(text="""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin-bottom: 10px;'>
    <h3 style='color: #c0c0c0; margin: 0 0 10px 0;'>La revelation du Train/Test Split</h3>
    <p style='color: #c0c0c0; margin: 0;'>
        <span style='color: #00FF00;'>L'erreur Train</span> diminue toujours quand le modele devient plus complexe.<br>
        <span style='color: #FF4444;'>L'erreur Test</span> diminue d'abord... puis EXPLOSE!<br><br>
        C'est la preuve que le modele a <b>memorise le bruit</b> au lieu d'apprendre le signal.<br>
        Un modele overfit performe bien sur les donnees qu'il connait, mais echoue sur de nouvelles donnees.
    </p>
</div>
""", width=PLOT_WIDTH)

layout4 = column(split_explanation, p4_data, p4_error)
show(layout4)

## Section 5 : Heatmap RSI - Sensibilite des Parametres

In [ ]:
# Telechargement des donnees BTC
print("Telechargement des donnees BTC...")
btc = vbt.YFData.download('BTC-USD', start='2020-01-01', end='2024-12-31').get('Close')
print(f"Donnees telechargees: {len(btc)} jours")

In [ ]:
# Parametres de la grille
rsi_windows = [5, 7, 10, 14, 20, 25, 30]
entry_thresholds = [20, 25, 30, 35, 40]

# Calcul de la heatmap
print("Calcul de la strategie RSI pour toutes les combinaisons...")

results = np.zeros((len(rsi_windows), len(entry_thresholds)))

for i, window in enumerate(rsi_windows):
    # Calcul du RSI
    rsi = vbt.RSI.run(btc, window=window).rsi
    
    for j, threshold in enumerate(entry_thresholds):
        # Signaux d'entree/sortie
        entries = rsi < threshold
        exits = rsi > (100 - threshold)
        
        # Backtest
        portfolio = vbt.Portfolio.from_signals(
            close=btc,
            entries=entries,
            exits=exits,
            init_cash=10000,
            fees=0.001
        )
        
        # Sharpe ratio
        sharpe = portfolio.sharpe_ratio()
        results[i, j] = sharpe if not np.isnan(sharpe) else 0

print("Calcul termine!")

# Conversion en DataFrame
results_df = pd.DataFrame(results, index=rsi_windows, columns=entry_thresholds)
print("\nHeatmap des Sharpe Ratios:")
print(results_df.round(2))

In [ ]:
def create_heatmap(data, title, width=600, height=400):
    """Cree une heatmap Bokeh interactive."""
    rsi_windows = list(data.index)
    entry_thresholds = list(data.columns)
    
    # Preparer les donnees pour Bokeh
    x_vals = []
    y_vals = []
    values = []
    
    for i, window in enumerate(rsi_windows):
        for j, threshold in enumerate(entry_thresholds):
            x_vals.append(threshold)
            y_vals.append(window)
            values.append(data.iloc[i, j])
    
    source = ColumnDataSource(data=dict(
        x=x_vals,
        y=y_vals,
        values=values
    ))
    
    # Color mapper (Rouge-Jaune-Vert)
    palette = list(reversed(RdYlGn11))
    mapper = LinearColorMapper(
        palette=palette,
        low=min(values),
        high=max(values)
    )
    
    # Figure
    p = figure(
        title=title,
        x_axis_label="Seuil RSI d'entree",
        y_axis_label="Fenetre RSI",
        width=width, height=height,
        x_range=[str(t) for t in entry_thresholds],
        y_range=[str(w) for w in rsi_windows],
        tools="hover,pan,wheel_zoom,reset"
    )
    apply_dark_theme(p)
    
    # Rectangles pour la heatmap
    p.rect(
        x=[str(x) for x in x_vals],
        y=[str(y) for y in y_vals],
        width=1, height=1,
        source=source,
        fill_color={'field': 'values', 'transform': mapper},
        line_color=DARK_BG
    )
    
    # Color bar
    color_bar = ColorBar(
        color_mapper=mapper,
        label_standoff=12,
        title="Sharpe",
        title_text_color=TEXT_COLOR,
        major_label_text_color=TEXT_COLOR,
        background_fill_color=DARK_BG
    )
    p.add_layout(color_bar, 'right')
    
    # Hover
    p.hover.tooltips = [
        ("Fenetre RSI", "@y"),
        ("Seuil entree", "@x"),
        ("Sharpe Ratio", "@values{0.00}")
    ]
    
    return p

# Creation de la heatmap
p5 = create_heatmap(results_df, "Heatmap de Sensibilite - Strategie RSI sur BTC (2020-2024)")

# Explication
heatmap_explanation = Div(text="""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin-bottom: 10px;'>
    <h3 style='color: #c0c0c0; margin: 0 0 10px 0;'>Comment lire cette heatmap?</h3>
    <p style='color: #c0c0c0; margin: 0;'>
        <span style='color: #00FF00;'>Zone VERTE</span> = Bonne performance (Sharpe eleve)<br>
        <span style='color: #FF4444;'>Zone ROUGE</span> = Mauvaise performance (Sharpe faible ou negatif)<br><br>
        <b>Ce qu'il faut chercher:</b><br>
        - Une <span style='color: #00FF00;'>zone verte coherente</span> (voisins aussi en vert) = SIGNAL robuste<br>
        - Un <span style='color: #FF4444;'>point isole</span> (entoure de rouge) = probablement du BRUIT/OVERFIT
    </p>
</div>
""", width=600)

layout5 = column(heatmap_explanation, p5)
show(layout5)

## Section 6 : In-Sample vs Out-of-Sample

In [ ]:
# Split temporel
btc_is = btc['2020-01-01':'2022-12-31']  # In-Sample
btc_oos = btc['2023-01-01':'2024-12-31']  # Out-of-Sample

print(f"In-Sample: {len(btc_is)} jours (2020-2022)")
print(f"Out-of-Sample: {len(btc_oos)} jours (2023-2024)")

In [ ]:
def compute_strategy_grid(price_data, rsi_windows, entry_thresholds):
    """Calcule la grille de performance pour une strategie RSI."""
    results = np.zeros((len(rsi_windows), len(entry_thresholds)))
    
    for i, window in enumerate(rsi_windows):
        rsi = vbt.RSI.run(price_data, window=window).rsi
        
        for j, threshold in enumerate(entry_thresholds):
            entries = rsi < threshold
            exits = rsi > (100 - threshold)
            
            portfolio = vbt.Portfolio.from_signals(
                close=price_data,
                entries=entries,
                exits=exits,
                init_cash=10000,
                fees=0.001
            )
            
            sharpe = portfolio.sharpe_ratio()
            results[i, j] = sharpe if not np.isnan(sharpe) else 0
    
    return pd.DataFrame(results, index=rsi_windows, columns=entry_thresholds)

# Calcul IS et OOS
print("Calcul In-Sample...")
results_is = compute_strategy_grid(btc_is, rsi_windows, entry_thresholds)

print("Calcul Out-of-Sample...")
results_oos = compute_strategy_grid(btc_oos, rsi_windows, entry_thresholds)

print("Calcul termine!")

In [ ]:
# Calculer les bornes communes pour une comparaison equitable
all_values = list(results_is.values.flatten()) + list(results_oos.values.flatten())
vmin = min(all_values)
vmax = max(all_values)

def create_heatmap_fixed_scale(data, title, vmin, vmax, width=450, height=350):
    """Cree une heatmap avec echelle de couleur fixe."""
    rsi_windows = list(data.index)
    entry_thresholds = list(data.columns)
    
    x_vals = []
    y_vals = []
    values = []
    
    for i, window in enumerate(rsi_windows):
        for j, threshold in enumerate(entry_thresholds):
            x_vals.append(threshold)
            y_vals.append(window)
            values.append(data.iloc[i, j])
    
    source = ColumnDataSource(data=dict(
        x=x_vals,
        y=y_vals,
        values=values
    ))
    
    palette = list(reversed(RdYlGn11))
    mapper = LinearColorMapper(
        palette=palette,
        low=vmin,
        high=vmax
    )
    
    p = figure(
        title=title,
        x_axis_label="Seuil RSI",
        y_axis_label="Fenetre RSI",
        width=width, height=height,
        x_range=[str(t) for t in entry_thresholds],
        y_range=[str(w) for w in rsi_windows],
        tools="hover,pan,wheel_zoom,reset"
    )
    apply_dark_theme(p)
    
    p.rect(
        x=[str(x) for x in x_vals],
        y=[str(y) for y in y_vals],
        width=1, height=1,
        source=source,
        fill_color={'field': 'values', 'transform': mapper},
        line_color=DARK_BG
    )
    
    color_bar = ColorBar(
        color_mapper=mapper,
        label_standoff=12,
        title="Sharpe",
        title_text_color=TEXT_COLOR,
        major_label_text_color=TEXT_COLOR,
        background_fill_color=DARK_BG
    )
    p.add_layout(color_bar, 'right')
    
    p.hover.tooltips = [
        ("Fenetre RSI", "@y"),
        ("Seuil", "@x"),
        ("Sharpe", "@values{0.00}")
    ]
    
    return p

# Creation des heatmaps
p_is = create_heatmap_fixed_scale(results_is, "IN-SAMPLE (2020-2022)", vmin, vmax)
p_oos = create_heatmap_fixed_scale(results_oos, "OUT-OF-SAMPLE (2023-2024)", vmin, vmax)

# Explication
isoos_explanation = Div(text="""
<div style='background: #2a2a2a; padding: 15px; border-radius: 5px; margin-bottom: 10px;'>
    <h3 style='color: #c0c0c0; margin: 0 0 10px 0;'>La preuve visuelle de l'overfitting</h3>
    <p style='color: #c0c0c0; margin: 0;'>
        Comparez les deux heatmaps avec la <b>meme echelle de couleur</b>:<br><br>
        <span style='color: #00FF00;'>Zones qui restent vertes</span> dans les deux = parametres ROBUSTES<br>
        <span style='color: #FF4444;'>Zones vertes IS mais rouges OOS</span> = OVERFITTING! Ces parametres ont juste eu de la chance.<br><br>
        C'est la claque emotionnelle: ce qui marchait "parfaitement" sur le passe peut s'effondrer sur le futur.
    </p>
</div>
""", width=900)

# Layout cote a cote
layout6 = column(isoos_explanation, row(p_is, p_oos))
show(layout6)

## Section 7 : Comment Eviter l'Overfitting

In [ ]:
# Bonnes pratiques visuelles
practices_html = """
<div style='padding: 20px;'>
    <h2 style='color: #00FF00; margin-bottom: 20px;'>Les Bonnes Pratiques</h2>
    
    <div style='display: flex; flex-wrap: wrap; gap: 20px;'>
        
        <div style='background: #2a2a2a; border-left: 4px solid #00FF00; padding: 15px; width: 400px; border-radius: 5px;'>
            <h3 style='color: #00FF00; margin: 0;'>1. Toujours faire un Train/Test Split</h3>
            <p style='color: #c0c0c0; margin-top: 10px;'>
                Gardez 20-30% de vos donnees de cote.<br>
                Ne les regardez JAMAIS pendant l'optimisation.<br>
                C'est votre "realite" pour valider.
            </p>
        </div>
        
        <div style='background: #2a2a2a; border-left: 4px solid #00FF00; padding: 15px; width: 400px; border-radius: 5px;'>
            <h3 style='color: #00FF00; margin: 0;'>2. Garder Peu de Parametres</h3>
            <p style='color: #c0c0c0; margin-top: 10px;'>
                Chaque parametre optimise = un degre de liberte pour overfitter.<br>
                Une strategie simple et robuste > une strategie complexe et fragile.
            </p>
        </div>
        
        <div style='background: #2a2a2a; border-left: 4px solid #00FF00; padding: 15px; width: 400px; border-radius: 5px;'>
            <h3 style='color: #00FF00; margin: 0;'>3. Chercher des Zones Robustes</h3>
            <p style='color: #c0c0c0; margin-top: 10px;'>
                Un bon parametre a des voisins qui performent aussi.<br>
                Si vous devez etre EXACT pour que ca marche, c'est du bruit.
            </p>
        </div>
        
        <div style='background: #2a2a2a; border-left: 4px solid #00FF00; padding: 15px; width: 400px; border-radius: 5px;'>
            <h3 style='color: #00FF00; margin: 0;'>4. Utiliser le Walk-Forward</h3>
            <p style='color: #c0c0c0; margin-top: 10px;'>
                Optimisez sur 2020, testez sur 2021.<br>
                Optimisez sur 2021, testez sur 2022.<br>
                Repetez pour voir la stabilite.
            </p>
        </div>
        
        <div style='background: #2a2a2a; border-left: 4px solid #00FF00; padding: 15px; width: 400px; border-radius: 5px;'>
            <h3 style='color: #00FF00; margin: 0;'>5. Tester sur d'Autres Actifs</h3>
            <p style='color: #c0c0c0; margin-top: 10px;'>
                Une strategie robuste devrait fonctionner sur ETH, SOL, etc.<br>
                Si elle ne marche QUE sur BTC, mefiez-vous.
            </p>
        </div>
        
    </div>
</div>
"""

practices_div = Div(text=practices_html, width=900)
show(practices_div)

## Section 8 : Comment Detecter l'Overfitting

In [ ]:
# Red flags visuels
redflags_html = """
<div style='padding: 20px;'>
    <h2 style='color: #FF4444; margin-bottom: 20px;'>Les Red Flags de l'Overfitting</h2>
    
    <div style='display: flex; flex-direction: column; gap: 15px;'>
        
        <div style='display: flex; align-items: center; padding: 15px; background: #2a2a2a; border-radius: 5px;'>
            <span style='color: #FF4444; font-size: 32px; margin-right: 20px;'>!</span>
            <div>
                <strong style='color: #FF4444; font-size: 18px;'>Sharpe Ratio > 3</strong>
                <p style='color: #c0c0c0; margin: 5px 0 0 0;'>
                    Les meilleurs hedge funds au monde font ~2. Si votre backtest fait 5+, c'est louche.
                </p>
            </div>
        </div>
        
        <div style='display: flex; align-items: center; padding: 15px; background: #2a2a2a; border-radius: 5px;'>
            <span style='color: #FF4444; font-size: 32px; margin-right: 20px;'>!</span>
            <div>
                <strong style='color: #FF4444; font-size: 18px;'>Point Isole sur la Heatmap</strong>
                <p style='color: #c0c0c0; margin: 5px 0 0 0;'>
                    Si le meilleur parametre est entoure de mauvais resultats, c'est probablement du bruit.
                </p>
            </div>
        </div>
        
        <div style='display: flex; align-items: center; padding: 15px; background: #2a2a2a; border-radius: 5px;'>
            <span style='color: #FF4444; font-size: 32px; margin-right: 20px;'>!</span>
            <div>
                <strong style='color: #FF4444; font-size: 18px;'>Trop de Parametres Optimises</strong>
                <p style='color: #c0c0c0; margin: 5px 0 0 0;'>
                    5+ parametres optimises? Vous avez probablement trouve du bruit, pas du signal.
                </p>
            </div>
        </div>
        
        <div style='display: flex; align-items: center; padding: 15px; background: #2a2a2a; border-radius: 5px;'>
            <span style='color: #FF4444; font-size: 32px; margin-right: 20px;'>!</span>
            <div>
                <strong style='color: #FF4444; font-size: 18px;'>Gros Ecart IS vs OOS</strong>
                <p style='color: #c0c0c0; margin: 5px 0 0 0;'>
                    Sharpe de 2.5 en IS et 0.3 en OOS? Classic overfit.
                </p>
            </div>
        </div>
        
        <div style='display: flex; align-items: center; padding: 15px; background: #2a2a2a; border-radius: 5px;'>
            <span style='color: #FF4444; font-size: 32px; margin-right: 20px;'>!</span>
            <div>
                <strong style='color: #FF4444; font-size: 18px;'>Winrate > 80%</strong>
                <p style='color: #c0c0c0; margin: 5px 0 0 0;'>
                    Meme les meilleurs traders font 55-60%. Un winrate trop eleve = suspicieux.
                </p>
            </div>
        </div>
        
        <div style='display: flex; align-items: center; padding: 15px; background: #2a2a2a; border-radius: 5px;'>
            <span style='color: #FF4444; font-size: 32px; margin-right: 20px;'>!</span>
            <div>
                <strong style='color: #FF4444; font-size: 18px;'>Drawdown Quasi Nul</strong>
                <p style='color: #c0c0c0; margin: 5px 0 0 0;'>
                    Toute strategie a des periodes difficiles. Pas de drawdown = backtest biaise.
                </p>
            </div>
        </div>
        
    </div>
    
    <div style='background: #2a2a2a; border: 2px solid #FFD700; padding: 20px; margin-top: 30px; border-radius: 5px;'>
        <h3 style='color: #FFD700; margin: 0 0 10px 0;'>La regle d'or</h3>
        <p style='color: #c0c0c0; margin: 0; font-size: 18px;'>
            Si ca semble trop beau pour etre vrai... <span style='color: #FF4444;'>c'est probablement le cas.</span>
        </p>
    </div>
</div>
"""

redflags_div = Div(text=redflags_html, width=900)
show(redflags_div)